In [11]:
# ==============================================================================
# UNIFIED PRODUCTION ENGINE: SYNCHRONOUS JUMP-DIFFUSION GRAPH NEURAL ODE
# ==============================================================================

import os
import gc
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from scipy.spatial import KDTree
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ==============================================================================
# 0. CONFIGURATION & NEW CITY DATA PATH ANCHORS (PLACEHOLDERS)
# ==============================================================================
# Swap these strings out whenever deploying the network to a brand new city canvas.

# Setup your project or system cluster context root path mapping
WORKSPACE_ROOT = Path("/home/jupyter-1nt23cb058/Capstone") 

CITY_DATA_PATHS = {
    # 1. Path to the PyTorch Geometric (.pt) global network topology graph object
    "global_topology_graph": WORKSPACE_ROOT / "data/processed/graph/topology_graph_pyg_inference.pt",
    
    # 2. Path to the parquet mapping table translating node IDs to internal index arrays
    "node_index_mapping": WORKSPACE_ROOT / "data/processed/graph/topology_nodeid_to_index_map.parquet",
    
    # 3. Path to the coordinate geometry sheet containing absolute spatial metrics (e.g., UTM x, y)
    "spatial_coordinates_sheet": WORKSPACE_ROOT / "data/graphs/bangalore_utm_nodes.parquet",
    
    # 4. Path to the multi-channel scale-normalized continuous master timeline dataframe 
    "master_scaled_timeline": WORKSPACE_ROOT / "data/processed/graph/master_scaled_checkpoint.parquet",
    
    # 5. Path to the final model input tracking features parquet file housing true ground-truth labels
    "model_hourly_features": WORKSPACE_ROOT / "data/processed/model_input/model_input_node_hourly_features.parquet"
}

# Core Structural Hyperparameters
NUM_CLUSTERS = 64
HIDDEN_DIM = 64
TARGET_SCALE = 342.9356  # Invariant scale factor balancing target extremes

# Establish workspace alignment path context safely
if WORKSPACE_ROOT.exists():
    os.chdir(WORKSPACE_ROOT)
else:
    WORKSPACE_ROOT = Path(os.getcwd())

# --- ENVIRONMENT & DETERMINISM INITIALIZATION ---
try:
    from torchdiffeq import odeint_adjoint as odeint
except ImportError:
    print("[WARN] torchdiffeq missing. Executing automatic baseline installation...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchdiffeq"])
    from torchdiffeq import odeint_adjoint as odeint

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INIT] Core Compute Target Context: {device}")

# ==============================================================================
# 1. NEURAL JUMP-DIFFUSION MODEL DEFINITIONS
# ==============================================================================
class ContinuousWindAdvectionDrift(nn.Module):
    """
    Computes continuous space-time trajectory derivatives (dh/dt) guided
    by localized edge orientations and real-time wind vector mechanics.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.node_transform = nn.Linear(hidden_dim, hidden_dim)
        self.msg_transform = nn.Linear(hidden_dim + 1, hidden_dim)
        self.act = nn.GELU()
        
        self.edge_index = None
        self.edge_bearings = None 
        self.current_wind_dir_deg = None

    def forward(self, t, h):
        row, col = self.edge_index
        reverse_bearings = (self.edge_bearings + 180.0) % 360.0
        angular_diff = torch.abs(reverse_bearings - self.current_wind_dir_deg)
        angular_diff = torch.minimum(angular_diff, 360.0 - angular_diff)
        
        cone_angle = 75.0
        advection_weight = torch.clamp(1.0 - (angular_diff / (cone_angle / 2.0)), min=0.0).unsqueeze(-1)
        
        node_proj = self.node_transform(h)
        msg_input = torch.cat([node_proj[row], advection_weight], dim=-1)
        directed_messages = self.msg_transform(msg_input)
        
        dh_dt = torch.zeros_like(h)
        dh_dt.index_add_(0, col, directed_messages)
        return self.act(dh_dt)

class HourlyDiscreteJumpOperator(nn.Module):
    """
    Applies instant trajectory mutations to state maps at discrete hourly intervals
    to absorb localized environmental shock shifts seamlessly.
    """
    def __init__(self, hidden_dim, feature_dim):
        super().__init__()
        self.shock_gate = nn.Sequential(
            nn.Linear(hidden_dim + feature_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh()
        )

    def forward(self, h_trajectory, hourly_features):
        merged_context = torch.cat([h_trajectory, hourly_features], dim=-1)
        delta_h = self.shock_gate(merged_context)
        return h_trajectory + delta_h

class SynchronousGraphJumpDiffusionODE(nn.Module):
    def __init__(self, input_dim=16, hidden_dim=64, output_dim=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.continuous_drift = ContinuousWindAdvectionDrift(hidden_dim)
        self.discrete_jump = HourlyDiscreteJumpOperator(hidden_dim, input_dim)
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )
        self.register_buffer("t_span", torch.tensor([0.0, 1.0]))

    def forward(self, x_seq, edge_index, edge_bearings):
        x_seq = x_seq.squeeze(0)  # [T, N, F]
        T, N, F = x_seq.shape
        
        self.continuous_drift.edge_index = edge_index
        self.continuous_drift.edge_bearings = edge_bearings
        
        h = self.encoder(x_seq[0])
        for t in range(1, T):
            self.continuous_drift.current_wind_dir_deg = float(x_seq[t, 0, 6].item())
            h = odeint(
                self.continuous_drift, h, self.t_span, 
                method='rk4', options={'step_size': 0.25}
            )[1]
            h = self.discrete_jump(h, x_seq[t])
            
        return self.decoder(h).view(1, N)

# ==============================================================================
# 2. GLOBAL TOPOLOGY INTERACTION & PARTITIONING
# ==============================================================================
print("[DATA] Loading parameterized system topology configurations...")
pyg_graph = torch.load(CITY_DATA_PATHS["global_topology_graph"], weights_only=False)
edge_index_global = pyg_graph.edge_index.long().cpu()
edge_attr_global = pyg_graph.edge_attr.float().cpu()
train_mask_global = pyg_graph.train_mask.bool().cpu()
num_nodes_global = int(pyg_graph.num_nodes)

coords_aligned = pd.read_parquet(CITY_DATA_PATHS["node_index_mapping"]).merge(
    pd.read_parquet(CITY_DATA_PATHS["spatial_coordinates_sheet"])[["osmid", "x", "y"]], 
    left_on="node_id", right_on="osmid", how="inner"
).sort_values("node_index")
coords_np = coords_aligned[["x", "y"]].to_numpy(dtype="float32")

class SpatialPartition:
    def __init__(self, cid, node_ids, edge_index, edge_attr, train_mask):
        self.cid = int(cid)
        self.n_id = torch.as_tensor(node_ids, dtype=torch.long)
        self.edge_index = edge_index.long()
        self.edge_attr = edge_attr.float()
        self.train_mask = train_mask.bool()
        self.edge_bearings = None  
        self.global_feature_registry = None

print(f"[PROCESS] Segmenting graph sub-mesh footprints into {NUM_CLUSTERS} blocks...")
kmeans_engine = KMeans(n_clusters=NUM_CLUSTERS, random_state=GLOBAL_SEED, n_init=10)
cluster_labels = kmeans_engine.fit_predict(coords_np)

cluster_data = []
edge_index_np = edge_index_global.numpy()

for cid in range(NUM_CLUSTERS):
    n_ids_global = np.where(cluster_labels == cid)[0].astype(np.int64)
    if len(n_ids_global) == 0: continue
    keep_edges_mask = np.isin(edge_index_np[0], n_ids_global) & np.isin(edge_index_np[1], n_ids_global)
    sub_edge_index = edge_index_np[:, keep_edges_mask]
    local_index_map = {int(global_idx): i for i, global_idx in enumerate(n_ids_global.tolist())}
    reindexed_src = np.array([local_index_map[int(x)] for x in sub_edge_index[0]], dtype=np.int64)
    reindexed_dst = np.array([local_index_map[int(x)] for x in sub_edge_index[1]], dtype=np.int64)
    
    partition = SpatialPartition(
        cid=cid, node_ids=n_ids_global,
        edge_index=torch.tensor(np.stack([reindexed_src, reindexed_dst]), dtype=torch.long),
        edge_attr=edge_attr_global[torch.from_numpy(keep_edges_mask)],
        train_mask=train_mask_global[torch.as_tensor(n_ids_global)]
    )
    
    if partition.edge_index.shape[1] > 0:
        src_xy = coords_np[partition.n_id[partition.edge_index[0]].numpy()]
        dst_xy = coords_np[partition.n_id[partition.edge_index[1]].numpy()]
        dx = dst_xy[:, 0] - src_xy[:, 0]
        dy = dst_xy[:, 1] - src_xy[:, 1]
        partition.edge_bearings = torch.from_numpy((np.degrees(np.arctan2(dy, dx)) + 360.0) % 360.0).float().to(device)
    else:
        partition.edge_bearings = torch.zeros(0, dtype=torch.float32, device=device)
    cluster_data.append(partition)

# ==============================================================================
# 3. INDUCTIVE MANIFOLD PARSING & KD-TREE NEIGHBOR BACKFALL
# ==============================================================================
print("[DATA] Loading multi-channel scale-normalized dataframes...")
master_df = pd.read_parquet(CITY_DATA_PATHS["master_scaled_timeline"])
BASE_11 = [
    "pm2_5_scaled", "pm10_scaled", "nitrogen_dioxide_scaled", "sulphur_dioxide_scaled",
    "carbon_monoxide_scaled", "wind_speed_10m_scaled", "wind_direction_10m_scaled",
    "wind_gusts_10m_scaled", "temperature_2m_scaled", "relative_humidity_2m_scaled", "surface_pressure_scaled"
]
FEATURE_COLS_16 = BASE_11 + [BASE_11[i] for i in [2, 3, 0, 1, 4]]

global_feature_registry = {}
for node_idx, framework_group in master_df.groupby("node_index"):
    feature_matrix = framework_group.sort_values("time")[FEATURE_COLS_16].to_numpy(dtype=np.float32)
    if feature_matrix.shape[0] > 0:
        global_feature_registry[int(node_idx)] = torch.from_numpy(feature_matrix)

valid_node_indices = np.array(list(global_feature_registry.keys()), dtype=np.int64)
spatial_tree = KDTree(coords_np[valid_node_indices])
_, neighbor_positions = spatial_tree.query(coords_np)
nearest_valid_global_indices = valid_node_indices[neighbor_positions]

for global_idx in range(num_nodes_global):
    if global_idx not in global_feature_registry:
        global_feature_registry[global_idx] = global_feature_registry[int(nearest_valid_global_indices[global_idx])]

for c in cluster_data:
    c.global_feature_registry = global_feature_registry

del master_df
gc.collect()

# ==============================================================================
# 4. TIER-AWARE RESAMPLING & TIMELINE GENERATION
# ==============================================================================
print("[DATA] Mapping chronological data loading profiles...")
df_raw = pd.read_parquet(CITY_DATA_PATHS["model_hourly_features"])
if "node_index" not in df_raw.columns:
    node_map = pd.read_parquet(CITY_DATA_PATHS["node_index_mapping"])
    node_to_idx = dict(zip(node_map["node_id"].values, node_map["node_index"].values))
    df_raw["node_index"] = df_raw["node_id"].map(node_to_idx)

all_times = pd.Index(sorted(df_raw["timestamp"].unique()))
T_total = len(all_times)
time_codes = pd.Categorical(df_raw["timestamp"], categories=all_times, ordered=True).codes.astype(np.int32)
raw_ts_indices = np.ascontiguousarray(time_codes)
raw_node_indices = np.ascontiguousarray(df_raw["node_index"].values.astype(np.int64))
df_raw["target_scaled"] = pd.to_numeric(df_raw["station_pm25"], errors="coerce").fillna(0.0).astype(np.float32) / TARGET_SCALE
raw_targets = np.ascontiguousarray(df_raw["target_scaled"].values)

sort_order = np.lexsort((raw_node_indices, raw_ts_indices))
raw_ts_indices = raw_ts_indices[sort_order]
raw_node_indices = raw_node_indices[sort_order]
raw_targets = raw_targets[sort_order]
time_breaks = np.searchsorted(raw_ts_indices, np.arange(T_total + 1, dtype=np.int32), side="left")

del df_raw
gc.collect()

class TierAwareClusterDatasetV3(Dataset):
    def __init__(self, clusters, t0, t1, window=12, stride=24):
        self.clusters = clusters
        self.window = int(window)
        self.starts = list(range(int(t0), min(int(t1), T_total) - self.window, int(stride)))
        self.cluster_global = [np.asarray(c.n_id.cpu().numpy(), dtype=np.int64) for c in clusters]
        self.samples = [(c_idx, t_start, "station") for c_idx in range(len(clusters)) for t_start in self.starts]
        
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        c_idx, t0, mode = self.samples[idx]
        part = self.clusters[c_idx]
        n_ids = self.cluster_global[c_idx]
        num_nodes = n_ids.shape[0]
        x_win = np.zeros((self.window, num_nodes, 16), dtype=np.float32)
        y_win = np.zeros((num_nodes,), dtype=np.float32)
        
        for w in range(self.window):
            t_curr = t0 + w
            for local_i, global_node_id in enumerate(n_ids.tolist()):
                reg = part.global_feature_registry[global_node_id]
                x_win[w, local_i, :] = reg[t_curr % reg.shape[0]].numpy()
                
            if w == self.window - 1:
                t_target_safe = t_curr if t_curr < len(time_breaks) - 1 else (len(time_breaks) - 2)
                s_ptr = time_breaks[t_target_safe]
                e_ptr = time_breaks[t_target_safe + 1]
                if e_ptr > s_ptr:
                    t_nodes = raw_node_indices[s_ptr:e_ptr]
                    t_vals = raw_targets[s_ptr:e_ptr]
                    member = np.isin(t_nodes, n_ids, assume_unique=False)
                    if np.any(member):
                        g_sel = t_nodes[member]
                        loc = np.searchsorted(n_ids, g_sel)
                        ok = (loc >= 0) & (loc < num_nodes) & (n_ids[loc] == g_sel)
                        if np.any(ok): y_win[loc[ok]] = t_vals[member][ok]
        return torch.from_numpy(x_win), torch.from_numpy(y_win), c_idx, int(t0)

val_ds = TierAwareClusterDatasetV3(cluster_data, t0=87178, t1=87514, window=12, stride=24)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)

# ==============================================================================
# 5. CORE EXECUTION MODEL OPTIMIZATION PASSTHROUGH
# ==============================================================================
print("[INIT] Instantiating Jump-Diffusion Graph ODE Engine Parameters...")
model = SynchronousGraphJumpDiffusionODE(input_dim=16, hidden_dim=HIDDEN_DIM, output_dim=1).to(device)

NUM_EPOCHS = 3
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
criterion = nn.MSELoss()

print(f"[START] Executing optimization run over {NUM_EPOCHS} comprehensive epochs...")
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss, steps_optimized = 0.0, 0
    epoch_bar = tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    
    for xb, vyb, c_idx_tensor, _ in epoch_bar:
        target_cluster_id = int(c_idx_tensor[0].item())
        partition_block = cluster_data[target_cluster_id]
        
        local_edge_index = partition_block.edge_index.to(device)
        local_edge_bearings = partition_block.edge_bearings.to(device)
        xb_device = xb.to(device, non_blocking=True)
        vyb_device = vyb.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        predictions = model(x_seq=xb_device, edge_index=local_edge_index, edge_bearings=local_edge_bearings)
        
        loss = criterion(predictions, vyb_device.view(1, -1))
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        steps_optimized += 1
        if steps_optimized % 5 == 0:
            epoch_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Avg": f"{epoch_loss / steps_optimized:.4f}"})
            
    print(f"🌟 [EPOCH {epoch:02d}/{NUM_EPOCHS:02d}] Optimization Loss Surface: {epoch_loss / steps_optimized:.6f}")

print("\n" + "="*80)
print("🏆 PIPELINE LOCKED. READY FOR LARGE-SCALE PRODUCTION TRAINING RUNS.")
print("="*80)

[INIT] Core Compute Target Context: cuda
[DATA] Loading parameterized system topology configurations...
[PROCESS] Segmenting graph sub-mesh footprints into 64 blocks...
[DATA] Loading multi-channel scale-normalized dataframes...
[DATA] Mapping chronological data loading profiles...
[INIT] Instantiating Jump-Diffusion Graph ODE Engine Parameters...
[START] Executing optimization run over 3 comprehensive epochs...


Epoch 1/3:   0%|          | 0/896 [00:00<?, ?it/s]

🌟 [EPOCH 01/03] Optimization Loss Surface: 808.529245


Epoch 2/3:   0%|          | 0/896 [00:00<?, ?it/s]

🌟 [EPOCH 02/03] Optimization Loss Surface: 0.001081


Epoch 3/3:   0%|          | 0/896 [00:00<?, ?it/s]

🌟 [EPOCH 03/03] Optimization Loss Surface: 0.000043

🏆 PIPELINE LOCKED. READY FOR LARGE-SCALE PRODUCTION TRAINING RUNS.


In [12]:
# ==============================================================================
# PHASE 7: ATOMIC SYSTEM WEIGHTS SERIALIZATION & ARCHIVING
# ==============================================================================

# 1. Define the destination archive paths
output_model_dir = Path("Capstone/notebooks/results/saved_models")
os.makedirs(output_model_dir, exist_ok=True)

stable_checkpoint_filename = "citywide_jump_diffusion_graph_ode_STABLE.pt"
checkpoint_save_path = output_model_dir / stable_checkpoint_filename

print(f"[SAVE] Serializing production hidden states to: '{checkpoint_save_path}'...")

# 2. Compile the mathematical state payload
# This preserves all neural coefficients, optimization velocity, and invariant constants
system_state_payload = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": int(NUM_EPOCHS),
    "final_loss": float(epoch_loss / steps_optimized),
    "timestamp": time.ctime(),
    "global_config": {
        "num_clusters": int(NUM_CLUSTERS),
        "hidden_dim": int(HIDDEN_DIM),
        "target_scale": float(TARGET_SCALE)
    }
}

# 3. Write atomic weights directly out to disk storage
torch.save(system_state_payload, checkpoint_save_path)

print(f"🏆 [SUCCESS] Weights archival locked successfully. File size: {checkpoint_save_path.stat().st_size / (1024*1024):.2f} MB")

[SAVE] Serializing production hidden states to: 'Capstone/notebooks/results/saved_models/citywide_jump_diffusion_graph_ode_STABLE.pt'...
🏆 [SUCCESS] Weights archival locked successfully. File size: 0.39 MB


In [13]:
# To reload the weights later:
checkpoint = torch.load(checkpoint_save_path, map_location=device)

# Load the parameters back into your instantiated model architecture
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("✅ Jump-Diffusion Graph ODE weights successfully reloaded for deployment.")

✅ Jump-Diffusion Graph ODE weights successfully reloaded for deployment.


In [14]:
# ==============================================================================
# INTEGRATION PHASE: SYNCHRONOUS JUMP-DIFFUSION GRAPH NEURAL ODE 
# ==============================================================================

from torchdiffeq import odeint_adjoint as odeint

class ContinuousWindAdvectionODEDrift(nn.Module):
    """
    Computes continuous space-time trajectory derivatives (dh/dt) guided
    by localized edge orientations and real-time wind vector mechanics.
    """
    def __init__(self, hidden_dim, edge_dim):
        super().__init__()
        self.node_transform = nn.Linear(hidden_dim, hidden_dim)
        self.msg_transform = nn.Linear(hidden_dim + edge_dim + 1, hidden_dim)
        self.act = nn.GELU()
        
        # Invariants populated dynamically by the parent forward method
        self.edge_index = None
        self.edge_attr = None
        self.upwind_mask = None

    def forward(self, t, h):
        row, col = self.edge_index
        
        # Isolate indices within the dynamic physical upwind cone radius
        u_mask = self.upwind_mask
        
        # Scale spatial gradients natively to match physics-informed structural requirements
        node_proj = self.node_transform(h)
        
        # Construct message payload containing source node space, static attributes, and upwind weights
        advection_weight = u_mask.float().unsqueeze(-1)
        msg_input = torch.cat([node_proj[row], self.edge_attr, advection_weight], dim=-1)
        directed_messages = self.msg_transform(msg_input)
        
        # Accumulate fluid boundary updates along destination target indices
        dh_dt = torch.zeros_like(h)
        dh_dt.index_add_(0, col, directed_messages)
        return self.act(dh_dt)

class HourlyDiscreteJumpOperator(nn.Module):
    """
    Applies instant trajectory mutations to state maps at discrete hourly intervals
    to absorb localized environmental shock shifts seamlessly.
    """
    def __init__(self, hidden_dim, feature_dim):
        super().__init__()
        self.shock_gate = nn.Sequential(
            nn.Linear(hidden_dim + feature_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh()
        )

    def forward(self, h_trajectory, hourly_features):
        merged_context = torch.cat([h_trajectory, hourly_features], dim=-1)
        delta_h = self.shock_gate(merged_context)
        return h_trajectory + delta_h

class ST_JumpDiffusion_GNodeODE(nn.Module):
    def __init__(self, node_in_dim=16, edge_dim=3, hidden_dim=64, output_dim=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.encoder = nn.Linear(node_in_dim, hidden_dim)
        self.continuous_drift = ContinuousWindAdvectionODEDrift(hidden_dim, edge_dim)
        self.discrete_jump = HourlyDiscreteJumpOperator(hidden_dim, node_in_dim)
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )
        self.register_buffer("t_span", torch.tensor([0.0, 1.0]))

    def forward(self, x_seq, edge_index, edge_attr, partition_obj):
        # Drop incoming batch dimension wrapper safely: [B=1, T=12, N, F=16] -> [T, N, F]
        x_seq = x_seq.squeeze(0)
        T, N, F = x_seq.shape
        
        # Sync topological context parameters
        self.continuous_drift.edge_index = edge_index
        self.continuous_drift.edge_attr = edge_attr
        
        # Base state initialization from the initial historical condition
        h = self.encoder(x_seq[0])
        
        for t in range(1, T):
            # Evaluate the upwind vector field mask natively on the GPU at each hour mark
            current_wind_dir_deg = float(x_seq[t, 0, 6].item())
            self.continuous_drift.upwind_mask = compute_dynamic_upwind_mask_vectorized(partition_obj, current_wind_dir_deg)
            
            # Continuous integration path pass over the hourly step horizon (O(1) memory via adjoint)
            h = odeint(
                self.continuous_drift, h, self.t_span, 
                method='rk4', options={'step_size': 0.25}
            )[1]
            
            # Discrete hourly correction step mutation
            h = self.discrete_jump(h, x_seq[t])
            
        return self.decoder(h).view(1, N)

print("[COMPLETE] Integration blueprint for ST_JumpDiffusion_GNodeODE compiled.")

[COMPLETE] Integration blueprint for ST_JumpDiffusion_GNodeODE compiled.


In [ ]:
# ==============================================================================
# PHASE 7 (INTEGRATED): SIDE-BY-SIDE HIGH-THROUGHPUT COMPARATIVE ENGINE (PATCHED)
# ==============================================================================

import os
import sys
import time
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

print("[INIT] Launching Integrated Side-by-Side Phase 7 Evaluation Core...")

# 1. Self-Healing Directory & Module Imports Alignment
ROOT_DIR = Path("/home/jupyter-1nt23cb058/Capstone")
if ROOT_DIR.exists():
    os.chdir(ROOT_DIR)
    if str(ROOT_DIR) not in sys.path:
        sys.path.insert(0, str(ROOT_DIR))
else:
    ROOT_DIR = Path(os.getcwd())
    if str(ROOT_DIR) not in sys.path:
        sys.path.insert(0, str(ROOT_DIR))

# Dynamic module layout import fallback path
try:
    from gnn.model import STPIGNN
    print("✅ Successfully pulled STPIGNN module reference from local architecture path.")
except ImportError as e:
    raise ImportError(f"Critical Framework Missing: Could not load STPIGNN from gnn.model. Error: {e}")

# 2. Workspace Constants & Shared Parameter Anchors
TARGET_SCALE = 342.9356  # Hard-preserved invariant scale balancing wildfire extremes

GLOBAL_CFG = {
    "lr_initial": 3e-6,
    "max_epochs": 15,
    "train_stride": 128,
    "val_stride": 24,
    "spatial_hidden_dim": 96,
    "temporal_hidden_dim": 96,
    "gnn_layers": 2,
    "physics_lambda_max": 0.060,
}

MAX_EPOCHS = int(GLOBAL_CFG["max_epochs"])
best_val_loss_pignn = float("inf")
best_val_loss_graph_ode = float("inf")

# ==============================================================================
# 3. SELF-HEALING DATA HOOK: RE-INITIALIZE DATASETS AND LOADERS IF OMITTED
# ==============================================================================
if 'train_loader' not in globals() or 'val_loader' not in globals():
    print("[WARN] Active data loaders not detected in memory. Initializing Phase 5 foundations...")
    
    df_raw_path = ROOT_DIR / "data/processed/model_input/model_input_node_hourly_features.parquet"
    if not df_raw_path.exists():
        raise FileNotFoundError(f"Required model input features parquet missing: {df_raw_path}")
        
    df_raw = pd.read_parquet(df_raw_path)
    ts_col = "timestamp" if "timestamp" in df_raw.columns else ("time" if "time" in df_raw.columns else "time")
    
    all_times = pd.Index(sorted(df_raw[ts_col].unique()))
    T_total = len(all_times)
    time_codes = pd.Categorical(df_raw[ts_col], categories=all_times, ordered=True).codes.astype(np.int32)
    
    raw_ts_indices = np.ascontiguousarray(time_codes)
    if "node_index" not in df_raw.columns:
        node_map = pd.read_parquet(ROOT_DIR / "data/processed/graph/topology_nodeid_to_index_map.parquet")
        node_to_idx = dict(zip(node_map["node_id"].values, node_map["node_index"].values))
        df_raw["node_index"] = df_raw["node_id"].map(node_to_idx)
        
    raw_node_indices = np.ascontiguousarray(df_raw["node_index"].values.astype(np.int64))
    df_raw["target_scaled"] = pd.to_numeric(df_raw["station_pm25"], errors="coerce").fillna(0.0).astype(np.float32) / TARGET_SCALE
    raw_targets = np.ascontiguousarray(df_raw["target_scaled"].values)
    
    sort_order = np.lexsort((raw_node_indices, raw_ts_indices))
    raw_ts_indices = raw_ts_indices[sort_order]
    raw_node_indices = raw_node_indices[sort_order]
    raw_targets = raw_targets[sort_order]
    
    time_breaks = np.searchsorted(raw_ts_indices, np.arange(T_total + 1, dtype=np.int32), side="left")
    
    del df_raw
    gc.collect()
    
    class TierAwareClusterDatasetV3(Dataset):
        def __init__(self, clusters, t0, t1, window=12, stride=128, phys_time_subsample=8):
            self.clusters = clusters
            self.window = int(window)
            self.starts = list(range(int(t0), min(int(t1), T_total) - self.window, int(stride)))
            self.cluster_global = [np.asarray(c.n_id.cpu().numpy(), dtype=np.int64) for c in clusters]
            self.cluster_has_sensor = [bool(c.train_mask.any()) for c in clusters]
            self.phys_time_subsample = max(1, int(phys_time_subsample))
            self.samples = []
            
            for c_idx, n_ids in enumerate(self.cluster_global):
                has_sensor = self.cluster_has_sensor[c_idx]
                if has_sensor:
                    for t_start in self.starts:
                        target_overlap = False
                        t_eval = t_start + self.window - 1
                        if t_eval < len(time_breaks) - 1:
                            s_ptr = time_breaks[t_eval]
                            e_ptr = time_breaks[t_eval + 1]
                            if e_ptr > s_ptr:
                                t_nodes = raw_node_indices[s_ptr:e_ptr]
                                if np.isin(t_nodes, n_ids, assume_unique=False).any():
                                    target_overlap = True
                        if target_overlap:
                            self.samples.append((c_idx, t_start, "station"))
                else:
                    for k, t_start in enumerate(self.starts):
                        if (k % self.phys_time_subsample) == 0:
                            self.samples.append((c_idx, t_start, "phys"))
                            
        def __len__(self): return len(self.samples)
        def __getitem__(self, idx):
            c_idx, t0, mode = self.samples[idx]
            part = self.clusters[c_idx]
            n_ids = self.cluster_global[c_idx]
            num_nodes = n_ids.shape[0]
            x_win = np.zeros((self.window, num_nodes, 16), dtype=np.float32)
            y_win = np.zeros((num_nodes,), dtype=np.float32)
            
            for w in range(self.window):
                t_curr = t0 + w
                for local_i, global_node_id in enumerate(n_ids.tolist()):
                    node_registry = part.global_feature_registry[global_node_id]
                    x_win[w, local_i, :] = node_registry[t_curr % node_registry.shape[0]].numpy()
                    
                if mode == "station" and w == self.window - 1:
                    t_target_safe = t_curr if t_curr < len(time_breaks) - 1 else (len(time_breaks) - 2)
                    s_ptr = time_breaks[t_target_safe]
                    e_ptr = time_breaks[t_target_safe + 1]
                    if e_ptr > s_ptr:
                        t_nodes = raw_node_indices[s_ptr:e_ptr]
                        t_vals = raw_targets[s_ptr:e_ptr]
                        member = np.isin(t_nodes, n_ids, assume_unique=False)
                        if np.any(member):
                            g_sel = t_nodes[member]
                            loc = np.searchsorted(n_ids, g_sel)
                            ok = (loc >= 0) & (loc < num_nodes) & (n_ids[loc] == g_sel)
                            if np.any(ok): y_win[loc[ok]] = t_vals[member][ok]
            return torch.from_numpy(x_win), torch.from_numpy(y_win), c_idx, int(t0)

    train_ds = TierAwareClusterDatasetV3(cluster_data, t0=0, t1=87178, window=12, stride=GLOBAL_CFG["train_stride"])
    val_ds = TierAwareClusterDatasetV3(cluster_data, t0=87178, t1=87514, window=12, stride=GLOBAL_CFG["val_stride"])
    
    weights = np.ones(len(train_ds.samples), dtype=np.float64)
    phys_indices = [i for i, (_, _, m) in enumerate(train_ds.samples) if m == "phys"]
    station_count = len(train_ds.samples) - len(phys_indices)
    w_phys = (0.35 * station_count) / max(1e-9, 0.65 * len(phys_indices))
    weights[phys_indices] = w_phys
    
    balanced_sampler = WeightedRandomSampler(weights=torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)
    
    train_loader = DataLoader(train_ds, batch_size=1, sampler=balanced_sampler, num_workers=4, pin_memory=True, prefetch_factor=2)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True, prefetch_factor=2)
    print("✅ Data loaders successfully re-synchronized in active namespace.")

# Infer the dynamic topological edge dimensions from your partitions
edge_dim = 3
for c in cluster_data:
    if hasattr(c, 'edge_attr') and c.edge_attr.numel() > 0:
        edge_dim = int(c.edge_attr.shape[-1])
        break

# 4. Instantiate Both Model Parameters
print("[INIT] Compiling alternative architecture blocks side-by-side...")
model_pignn = STPIGNN(
    node_in_dim=16, edge_dim=edge_dim,
    spatial_hidden_dim=GLOBAL_CFG["spatial_hidden_dim"],
    temporal_hidden_dim=GLOBAL_CFG["temporal_hidden_dim"],
    gnn_layers=GLOBAL_CFG["gnn_layers"]
).to(device)

model_ode = ST_JumpDiffusion_GNodeODE(
    node_in_dim=16, edge_dim=edge_dim,
    hidden_dim=GLOBAL_CFG["spatial_hidden_dim"]
).to(device)

# Set up isolated optimization targets
optimizer_pignn = torch.optim.AdamW(model_pignn.parameters(), lr=GLOBAL_CFG["lr_initial"], weight_decay=1e-2)
optimizer_ode = torch.optim.AdamW(model_ode.parameters(), lr=GLOBAL_CFG["lr_initial"], weight_decay=1e-2)

amp_enabled = (device.type == "cuda")
scaler_amp = torch.amp.GradScaler("cuda", enabled=amp_enabled)

# 5. Core Metric Calculators
def masked_mae_rmse_vectorized(pred, target, mask):
    pred_1d, target_1d, mask_1d = pred.view(-1), target.view(-1), mask.view(-1)
    if not mask_1d.any():
        zero_t = pred.new_tensor(0.0)
        return zero_t, zero_t
    errors = pred_1d[mask_1d] - target_1d[mask_1d]
    return torch.mean(torch.abs(errors)), torch.sqrt(torch.mean(errors * errors))

def compute_wind_advection_penalty(pred, edge_index, upwind_edge_mask, edge_attr):
    if edge_index.shape[1] == 0 or not upwind_edge_mask.any():
        return pred.new_tensor(0.0)
    row, col = edge_index
    spatial_diff = pred[col[upwind_edge_mask]] - pred[row[upwind_edge_mask]]
    if edge_attr is not None and edge_attr.shape[0] == edge_index.shape[1]:
        normalized_attr = edge_attr[upwind_edge_mask] / 1000.0
        resistance = torch.sigmoid(normalized_attr.sum(dim=-1))
        spatial_diff = spatial_diff * resistance
    return torch.mean(F.relu(spatial_diff) ** 2)

# ==============================================================================
# 6. TRAINING EXECUTION MATRIX
# ==============================================================================
try:
    for epoch in range(1, MAX_EPOCHS + 1):
        curr_lambda = float(GLOBAL_CFG["physics_lambda_max"]) * min(1.0, epoch / 25.0)
        
        loss_track = {"pignn": 0.0, "graph_ode": 0.0}
        mae_track = {"pignn": 0.0, "graph_ode": 0.0}
        total_train_batches = 0
        
        model_pignn.train()
        model_ode.train()
        
        batch_bar = tqdm(total=len(train_loader), desc=f"Epoch {epoch} Dual-Training", position=0, leave=True)
        
        for i, (xb, yb, c_idx_t, _) in enumerate(train_loader):
            c_idx = int(c_idx_t[0].item())
            part = cluster_data[c_idx]
            
            xb = xb.to(device, non_blocking=True)
            yb = yb.squeeze(dim=0).to(device, non_blocking=True).view(-1)
            edge_i = part.edge_index.to(device, non_blocking=True)
            edge_a = part.edge_attr.to(device, non_blocking=True)
            
            base_mask = part.train_mask.to(device=device, dtype=torch.bool).view(-1)
            sup_mask = base_mask & (torch.abs(yb) > 1e-8)
            
            current_wind_dir_deg = float(xb[0, 0, 0, 6].item())
            u_mask = compute_dynamic_upwind_mask_local(part, current_wind_dir_deg)
            
            # --- STREAM A: STPIGNN ---
            optimizer_pignn.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                pred_pignn = model_pignn(x_seq=xb, edge_index=edge_i, edge_attr=edge_a).view(-1)
                if bool(sup_mask.any()):
                    data_loss = torch.sum((pred_pignn - yb) ** 2 * sup_mask.float()) / torch.clamp(sup_mask.sum(), min=1.0)
                    phys_penalty = compute_wind_advection_penalty(pred_pignn, edge_i, u_mask, edge_a)
                    total_loss_pignn = data_loss + (curr_lambda * phys_penalty)
                else:
                    phys_penalty = compute_wind_advection_penalty(pred_pignn, edge_i, u_mask, edge_a)
                    total_loss_pignn = curr_lambda * phys_penalty
            
            if torch.isfinite(total_loss_pignn) and total_loss_pignn.detach() > 0.0:
                scaler_amp.scale(total_loss_pignn).backward()
                scaler_amp.unscale_(optimizer_pignn)
                torch.nn.utils.clip_grad_norm_(model_pignn.parameters(), max_norm=0.3)
                scaler_amp.step(optimizer_pignn)
                
            # --- STREAM B: Graph Jump-Diffusion ODE ---
            optimizer_ode.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                pred_ode = model_ode(x_seq=xb, edge_index=edge_i, edge_attr=edge_a, partition_obj=part).view(-1)
                if bool(sup_mask.any()):
                    data_loss_ode = torch.sum((pred_ode - yb) ** 2 * sup_mask.float()) / torch.clamp(sup_mask.sum(), min=1.0)
                    phys_penalty_ode = compute_wind_advection_penalty(pred_ode, edge_i, u_mask, edge_a)
                    total_loss_ode = data_loss_ode + (curr_lambda * phys_penalty_ode)
                else:
                    phys_penalty_ode = compute_wind_advection_penalty(pred_ode, edge_i, u_mask, edge_a)
                    total_loss_ode = curr_lambda * phys_penalty_ode
                    
            if torch.isfinite(total_loss_ode) and total_loss_ode.detach() > 0.0:
                scaler_amp.scale(total_loss_ode).backward()
                scaler_amp.unscale_(optimizer_ode)
                torch.nn.utils.clip_grad_norm_(model_ode.parameters(), max_norm=0.3)
                scaler_amp.step(optimizer_ode)
                
            scaler_amp.update()
            
            with torch.no_grad():
                mae_pignn, _ = masked_mae_rmse_vectorized(pred_pignn, yb, sup_mask)
                mae_ode, _ = masked_mae_rmse_vectorized(pred_ode, yb, sup_mask)
                
            loss_track["pignn"] += total_loss_pignn.detach().item()
            loss_track["graph_ode"] += total_loss_ode.detach().item()
            mae_track["pignn"] += mae_pignn.item()
            mae_track["graph_ode"] += mae_ode.item()
            total_train_batches += 1
            
            if i % 10 == 0:
                batch_bar.set_postfix({
                    "PIGNN_Loss": f"{loss_track['pignn']/total_train_batches:.4f}",
                    "ODE_Loss": f"{loss_track['graph_ode']/total_train_batches:.4f}",
                    "ODE_MAE": f"{mae_track['graph_ode']/total_train_batches:.4f}"
                })
            batch_bar.update(1)
            
        batch_bar.close()
        
        # --- INDUCTIVE OUT-OF-SAMPLES VALIDATION SWEEP ---
        model_pignn.eval()
        model_ode.eval()
        
        val_loss_pignn, val_loss_ode = 0.0, 0.0
        val_mae_pignn, val_mae_ode = 0.0, 0.0
        total_val_batches = 0
        
        with torch.no_grad():
            for vxb, vyb, vc_idx_t, _ in val_loader:
                vc_idx = int(vc_idx_t[0].item())
                vpart = cluster_data[vc_idx]
                
                vxb = vxb.to(device, non_blocking=True)
                vyb = vyb.squeeze(dim=0).to(device, non_blocking=True).view(-1)
                v_edge_i = vpart.edge_index.to(device, non_blocking=True)
                v_edge_a = vpart.edge_attr.to(device, non_blocking=True)
                
                v_base_mask = vpart.train_mask.to(device=device, dtype=torch.bool).view(-1)
                v_sup_mask = v_base_mask & (torch.abs(vyb) > 1e-8)
                
                v_wind_dir_deg = float(vxb[0, 0, 0, 6].item())
                u_mask = compute_dynamic_upwind_mask_local(vpart, v_wind_dir_deg)
                
                with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                    vp_pignn = model_pignn(x_seq=vxb, edge_index=v_edge_i, edge_attr=v_edge_a).view(-1)
                    vp_ode = model_ode(x_seq=vxb, edge_index=v_edge_i, edge_attr=v_edge_a, partition_obj=vpart).view(-1)
                    
                    v_loss_pignn = torch.sum((vp_pignn - vyb) ** 2 * v_sup_mask.float()) / torch.clamp(v_sup_mask.sum(), min=1.0)
                    v_loss_ode = torch.sum((vp_ode - vyb) ** 2 * v_sup_mask.float()) / torch.clamp(v_sup_mask.sum(), min=1.0)
                
                v_mae_pignn, _ = masked_mae_rmse_vectorized(vp_pignn, vyb, v_sup_mask)
                v_mae_ode, _ = masked_mae_rmse_vectorized(vp_ode, vyb, v_sup_mask)
                
                val_loss_pignn += v_loss_pignn.item()
                val_loss_ode += v_loss_ode.item()
                val_mae_pignn += v_mae_pignn.item()
                val_mae_ode += v_mae_ode.item()
                total_val_batches += 1
                
        final_pignn_mae = (val_mae_pignn / total_val_batches) * TARGET_SCALE
        final_ode_mae = (val_mae_ode / total_val_batches) * TARGET_SCALE
        
        tqdm.write(
            f"📊 [EPOCH {epoch:02d}/{MAX_EPOCHS:02d}] "
            f"ST-PIGNN OOS MAE: {final_pignn_mae:.3f} ug/m³ || "
            f"Graph-ODE OOS MAE: {final_ode_mae:.3f} ug/m³"
        )
        
        if (val_loss_ode / total_val_batches) < best_val_loss_graph_ode:
            best_val_loss_graph_ode = val_loss_ode / total_val_batches
            torch.save(model_ode.state_dict(), "citywide_jump_diffusion_graph_ode_BEST.pt")
            tqdm.write(f"🏆 [JUMP-DIFFUSION CHAMPION] Lowered validation error to: {final_ode_mae:.4f} ug/m³")

except KeyboardInterrupt:
    print("[INTERRUPT] Graceful evaluation pause registered.")

[INIT] Launching Integrated Side-by-Side Phase 7 Evaluation Core...
✅ Successfully pulled STPIGNN module reference from local architecture path.
[WARN] Active data loaders not detected in memory. Initializing Phase 5 foundations...
✅ Data loaders successfully re-synchronized in active namespace.
[INIT] Compiling alternative architecture blocks side-by-side...


Epoch 1 Dual-Training:   0%|          | 0/14343 [00:00<?, ?it/s]

📊 [EPOCH 01/15] ST-PIGNN OOS MAE: 46.550 ug/m³ || Graph-ODE OOS MAE: nan ug/m³


Epoch 2 Dual-Training:   0%|          | 0/14343 [00:00<?, ?it/s]

📊 [EPOCH 02/15] ST-PIGNN OOS MAE: 42.469 ug/m³ || Graph-ODE OOS MAE: nan ug/m³


Epoch 3 Dual-Training:   0%|          | 0/14343 [00:00<?, ?it/s]

📊 [EPOCH 03/15] ST-PIGNN OOS MAE: 41.881 ug/m³ || Graph-ODE OOS MAE: nan ug/m³


Epoch 4 Dual-Training:   0%|          | 0/14343 [00:00<?, ?it/s]

📊 [EPOCH 04/15] ST-PIGNN OOS MAE: 42.037 ug/m³ || Graph-ODE OOS MAE: nan ug/m³


Epoch 5 Dual-Training:   0%|          | 0/14343 [00:00<?, ?it/s]